In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Conversión csv a dataframes de pandas

In [ ]:
ruta_per_capita = "/content/drive/MyDrive/Visualización de datos/DATASETS_RAW/annual-healthcare-expenditure-per-capita.csv"
ruta_mortalidad_inf = "/content/drive/MyDrive/Visualización de datos/DATASETS_RAW/child-mortality.csv"
ruta_diferencia_exp = "/content/drive/MyDrive/Visualización de datos/DATASETS_RAW/difference-in-female-and-male-life-expectancy-at-birth.csv"
ruta_calidad_acc = "/content/drive/MyDrive/Visualización de datos/DATASETS_RAW/healthcare-access-quality-un.csv"
ruta_camas_hos = "/content/drive/MyDrive/Visualización de datos/DATASETS_RAW/hospital-beds-per-1000-people.csv"
ruta_expectativa_sex = "/content/drive/MyDrive/Visualización de datos/DATASETS_RAW/life-expectation-at-birth-by-sex.csv"
ruta_mortalidad_mat = "/content/drive/MyDrive/Visualización de datos/DATASETS_RAW/maternal-mortality.csv"
ruta_oecd_financial = "/content/drive/MyDrive/Visualización de datos/DATASETS_RAW/oecd_financial.csv"

import pandas as pd

df_per_capita = pd.read_csv(ruta_per_capita)
df_mortalidad_inf = pd.read_csv(ruta_mortalidad_inf)
df_diferencia_exp = pd.read_csv(ruta_diferencia_exp)
df_calidad_acc = pd.read_csv(ruta_calidad_acc)
df_camas_hos = pd.read_csv(ruta_camas_hos)
df_expectativa_sex = pd.read_csv(ruta_expectativa_sex)
df_mortalidad_mat = pd.read_csv(ruta_mortalidad_mat)
df_oecd_financial = pd.read_csv(ruta_oecd_financial)

#Limpieza individual de datasets

In [ ]:
AÑO_MIN, AÑO_MAX = 2000, 2023

def limpiar_base(df, col_valor, nuevo_nombre):
    """
    filtra el rango de años, elimina filas sin código ISO
    (que corresponden a agregados como 'World', continentes, etc.)
    y renombra la columna de valor.
    """
    df = df[df['Year'].between(AÑO_MIN, AÑO_MAX)].copy()
    df = df[df['Code'].notna() & (df['Code'] != '')].copy()
    df = df.rename(columns={col_valor: nuevo_nombre})
    return df[['Entity', 'Code', 'Year', nuevo_nombre]]

# 1. gasto per cápita
df_per_capita = limpiar_base(
    df_per_capita,
    'Current health expenditure per capita, PPP (current international $)',
    'gasto_per_capita'
)

# 2. mortalidad infantil
df_mortalidad_inf = limpiar_base(
    df_mortalidad_inf,
    'Under-five mortality rate (selected)',
    'mortalidad_infantil'
)

# 3. brecha de esperanza de vida (F - M)
df_diferencia_exp = limpiar_base(
    df_diferencia_exp,
    'Life expectancy (female-male difference) at birth, period',
    'brecha_genero'
)

# 4. cobertura UHC
df_calidad_acc = limpiar_base(
    df_calidad_acc,
    'UHC service coverage index',
    'uhc_index'
)

# 5. camas hospitalarias
df_camas_hos = limpiar_base(
    df_camas_hos,
    'Hospital beds (per 1,000 people)',
    'camas_por_mil'
)

# 6. esperanza de vida por sexo (tiene 2 columnas de valor, caso especial)
df_exp = df_expectativa_sex.copy()
df_exp = df_exp[df_exp['Year'].between(AÑO_MIN, AÑO_MAX)]
df_exp = df_exp[df_exp['Code'].notna() & (df_exp['Code'] != '')]
df_exp = df_exp.rename(columns={'Female': 'exp_vida_mujer', 'Male': 'exp_vida_hombre'})
df_exp = df_exp[['Entity', 'Code', 'Year', 'exp_vida_mujer', 'exp_vida_hombre']]

# 7. mortalidad materna (conservamos la columna de región geográfica)
df_mortalidad_mat = df_mortalidad_mat[df_mortalidad_mat['Year'].between(AÑO_MIN, AÑO_MAX)]
df_mortalidad_mat = df_mortalidad_mat[df_mortalidad_mat['Code'].notna() & (df_mortalidad_mat['Code'] != '')]
df_mortalidad_mat = df_mortalidad_mat.rename(columns={
    'Maternal mortality ratio': 'mortalidad_materna',
    'World region according to OWID': 'region'
})
# descartamos la columna Annotations (casi completamente vacia)
df_mortalidad_mat = df_mortalidad_mat[['Entity', 'Code', 'Year', 'mortalidad_materna', 'region']]

# 8. OECD financiero (simplificamos las 46 columnas a solo 3 útiles)
df_oecd = df_oecd_financial[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']].copy()
df_oecd.columns = ['Code', 'Year', 'gasto_pct_pib']
df_oecd['Year'] = pd.to_numeric(df_oecd['Year'], errors='coerce')
df_oecd = df_oecd[df_oecd['Year'].between(AÑO_MIN, AÑO_MAX)]
df_oecd = df_oecd.dropna(subset=['Year', 'gasto_pct_pib'])
df_oecd['Year'] = df_oecd['Year'].astype(int)

for nombre, df in [('per_capita', df_per_capita), ('mort_inf', df_mortalidad_inf), ('brecha', df_diferencia_exp),
                   ('uhc', df_calidad_acc), ('camas', df_camas_hos), ('exp_vida', df_exp),
                   ('mort_mat', df_mortalidad_mat), ('oecd', df_oecd)]:
    print(f"  {nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")

  per_capita: 4846 filas, 4 columnas
  mort_inf: 5099 filas, 4 columnas
  brecha: 5712 filas, 4 columnas
  uhc: 4920 filas, 4 columnas
  camas: 3189 filas, 4 columnas
  exp_vida: 5904 filas, 5 columnas
  mort_mat: 4157 filas, 5 columnas
  oecd: 461 filas, 3 columnas


#Unificación del dataset maestro

In [ ]:
df_maestro = (
    df_per_capita
    .merge(df_mortalidad_inf,     on=['Entity', 'Code', 'Year'], how='outer')
    .merge(df_diferencia_exp,  on=['Entity', 'Code', 'Year'], how='outer')
    .merge(df_calidad_acc,     on=['Entity', 'Code', 'Year'], how='outer')
    .merge(df_camas_hos,   on=['Entity', 'Code', 'Year'], how='outer')
    .merge(df_exp,     on=['Entity', 'Code', 'Year'], how='outer')
    .merge(df_mortalidad_mat,      on=['Entity', 'Code', 'Year'], how='outer')
    .merge(df_oecd,    on=['Code', 'Year'],           how='left')
)

df_maestro = df_maestro.sort_values(['Entity', 'Year']).reset_index(drop=True)

##Agregado de regiones y eliminación de agregados regionales y economicos

In [ ]:
# rellenar región de paises
df_maestro['region'] = df_maestro.groupby('Code')['region'].transform('first')

print(f"Dataset maestro: {df_maestro.shape[0]} filas × {df_maestro.shape[1]} columnas ✅")
print("\nColumnas:", df_maestro.columns.tolist())
print("\nVista previa:")
df_maestro.head(800)

#eliminar agregados
agregados = [
    'Africa', 'Asia', 'Europe', 'Oceania', 'North America', 'South America',
    'World', 'European Union (27)',
    'East Asia and Pacific (WB)', 'Europe and Central Asia (WB)',
    'Latin America and Caribbean (WB)', 'Middle East, North Africa, Afghanistan and Pakistan (WB)',
    'North America (WB)', 'South Asia (WB)', 'Sub-Saharan Africa (WB)',
    'High-income countries', 'Low-income countries',
    'Lower-middle-income countries', 'Upper-middle-income countries'
]

df_maestro = df_maestro[~df_maestro['Entity'].isin(agregados)].copy()

#agregar regiones manualmente
regiones_manuales = {
    # Europa
    'Andorra': 'Europe', 'Monaco': 'Europe', 'San Marino': 'Europe',
    'Liechtenstein': 'Europe', 'Kosovo': 'Europe', 'Gibraltar': 'Europe',
    'Faroe Islands': 'Europe', 'Guernsey': 'Europe', 'Jersey': 'Europe',
    'Isle of Man': 'Europe', 'Vatican': 'Europe',
    # Asia / Oceanía
    'Taiwan': 'Asia', 'Macao': 'Asia',
    'Cook Islands': 'Oceania', 'Niue': 'Oceania', 'Nauru': 'Oceania',
    'Palau': 'Oceania', 'Tokelau': 'Oceania', 'Tuvalu': 'Oceania',
    'Marshall Islands': 'Oceania', 'New Caledonia': 'Oceania',
    'French Polynesia': 'Oceania', 'Wallis and Futuna': 'Oceania',
    'Northern Mariana Islands': 'Oceania', 'Guam': 'Oceania',
    'American Samoa': 'Oceania',
    # América
    'Aruba': 'America', 'Curacao': 'America', 'Bermuda': 'America',
    'Cayman Islands': 'America', 'Anguilla': 'America',
    'British Virgin Islands': 'America', 'Turks and Caicos Islands': 'America',
    'Montserrat': 'America', 'Saint Kitts and Nevis': 'America',
    'Saint Barthelemy': 'America', 'Saint Martin (French part)': 'America',
    'Sint Maarten (Dutch part)': 'America', 'Saint Pierre and Miquelon': 'America',
    'Saint Helena': 'America', 'Falkland Islands': 'America',
    'French Guiana': 'America', 'Guadeloupe': 'America',
    'Martinique': 'America',
    'United States Virgin Islands': 'America',
    # África
    'Western Sahara': 'Africa',
    'Bonaire Sint Eustatius and Saba': 'Africa',
    # Otros
    'Greenland': 'Europe',
}

df_maestro['region'] = df_maestro.apply(
    lambda row: regiones_manuales.get(row['Entity'], row['region']),
    axis=1
)

#unificar ramificaciones de america
df_maestro['region'] = df_maestro['region'].replace({
    'North America': 'America',
    'South America': 'America'
})


Dataset maestro: 6144 filas × 13 columnas ✅

Columnas: ['Entity', 'Code', 'Year', 'gasto_per_capita', 'mortalidad_infantil', 'brecha_genero', 'uhc_index', 'camas_por_mil', 'exp_vida_mujer', 'exp_vida_hombre', 'mortalidad_materna', 'region', 'gasto_pct_pib']

Vista previa:


# Análisis del dataset

In [ ]:
print(f"\nDataset maestro: {df_maestro.shape[0]:,} filas × {df_maestro.shape[1]} columnas")
print(f"Países únicos:   {df_maestro['Code'].nunique()}")
print(f"Años cubiertos:  {df_maestro['Year'].min()} – {df_maestro['Year'].max()}")

print("\n% de nulos por columna:")
nulos = (df_maestro.isnull().sum() / len(df_maestro) * 100).round(1)
for col, pct in nulos.items():
    barra = '█' * int(pct / 5)
    print(f"  {col:<45} {pct:5.1f}%  {barra}")

print("\n¿Cuántos países tienen al menos un valor en gasto_pct_pib (OECD)?")
print(df_maestro[df_maestro['gasto_pct_pib'].notna()]['Code'].nunique(), "países")

print("\nVista previa (primeras 5 filas):")
df_maestro.head(5)


Dataset maestro: 5,688 filas × 13 columnas
Países únicos:   237
Años cubiertos:  2000 – 2023

% de nulos por columna:
  Entity                                          0.0%  
  Code                                            0.0%  
  Year                                            0.0%  
  gasto_per_capita                               20.3%  ████
  mortalidad_infantil                            15.4%  ███
  brecha_genero                                   0.0%  
  uhc_index                                      18.6%  ███
  camas_por_mil                                  48.2%  █████████
  exp_vida_mujer                                  0.0%  
  exp_vida_hombre                                 0.0%  
  mortalidad_materna                             31.0%  ██████
  region                                          0.0%  
  gasto_pct_pib                                  91.9%  ██████████████████

¿Cuántos países tienen al menos un valor en gasto_pct_pib (OECD)?
52 países

Vista previa (prime

,Entity,Code,Year,gasto_per_capita,mortalidad_infantil,brecha_genero,uhc_index,camas_por_mil,exp_vida_mujer,exp_vida_hombre,mortalidad_materna,region,gasto_pct_pib
0,Afghanistan,AFG,2000,NaN,13.17,3.091999,30.0,NaN,56.5547,53.4627,1346.1442,Asia,NaN
1,Afghanistan,AFG,2001,NaN,12.74,3.148102,31.0,NaN,57.0889,53.9408,1273.4314,Asia,NaN
2,Afghanistan,AFG,2002,85.85750,12.31,2.455898,32.0,NaN,57.4417,54.9858,1277.3080,Asia,NaN
3,Afghanistan,AFG,2003,85.93302,11.87,2.623501,33.0,NaN,58.4724,55.8489,1196.0907,Asia,NaN
4,Afghanistan,AFG,2004,93.93581,11.42,2.553799,34.0,NaN,59.0728,56.5190,1114.8872,Asia,NaN


In [ ]:
ruta_salida = '/content/drive/MyDrive/Visualización de datos/salud_maestro.csv'
df_maestro.to_csv(ruta_salida, index=False)

print(f"Archivo guardado en: {ruta_salida} ✅")

Archivo guardado en: /content/drive/MyDrive/Visualización de datos/DATASETS_RAW/salud_maestro.csv ✅
